In [1]:
# Replace with your name
print("Peter Bowman")

Prof. Brown


# Build a Chatbot with Neural Network

We've discovered how to build a chatbot with cosine similarity. Now, let's explore how we might build one with neural network!

We will create our training data, train a neural network with them, then use the trained model to make our chatbot.  

# 1. Install Libraries

Firstly, we will install libraries needed for this neural network powered chatbot.
Keras is a machine learning library which utilizes tensorflow (another lower level machine learning library) at the backend. This makes it easier for us to deploy deep neural network for this purpose.

In [3]:
# Task 1: Execute Library Imports
from tensorflow.keras.models import Sequential
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.layers import Dense

from numpy import argmax
import numpy as np
import re

# 2. Input training data

We will first include the following training data for our chatbot:
1. X represent the different possible inputs that users might enter
2. Y represent the intent of the inputs

In [4]:
# Task 2: Create the possible input array
X = ['Hi',
     'Hello',
     'How are you?',
     'I am making',
     'making',
     'working',
     'studying',
     'see you later',
     'bye',
     'goodbye']

print(len(X))

10


In [5]:
# Task 3: Create the intent of the inputs
Y = ['greeting',
     'greeting',
     'greeting',
     'busy',
     'busy',
     'busy',
     'busy',
     'bye',
     'bye',
     'bye']

print(len(Y))

10


Notice that there are several different sentences that have similar intent. Here, we are only having 3 intents, but you can add as many as you want for your project!

This is the way our chatbot will work:
1. From the input sentence, we will identify the intent using our trained AI model.
2. For each intent, we have a prepared response.

For example, if we identify that the intent of the input is for a greeting, we might ask the chatbot to reply with a greeting as well, something like 'hi' or 'how are you doing?'

We will use machine learning to create a model that can classify input sentence into different intents.
We make it as follows:

1. We create a training data (X and Y above) which contains a list of sentences and their intents.
2. Use the training data to train a classifier.
3. Vectorize input sentences and use classifier to determine intent.

# 3. Text processing

As usual, we will start with text processing. Do you remember the process?

## 3.1 Remove non alphanumeric characters

In [6]:
# Task 4: Text preprocessing

def remove_non_alpha_numeric_characters(sentence):
    new_sentence = ''
    for alphabet in sentence:
        if alphabet.isalpha() or alphabet == ' ':
            new_sentence += alphabet
    return new_sentence

def preprocess_data(X):
    X = [data_point.lower() for data_point in X]
    X = [remove_non_alpha_numeric_characters(
        sentence) for sentence in X]
    X = [data_point.strip() for data_point in X]
    X = [re.sub(' +', ' ',
                data_point) for data_point in X]
    return X

# Create Vocabulary
X = preprocess_data(X)

vocabulary = set()
for data_point in X:
    for word in data_point.split(' '):
        vocabulary.add(word)

vocabulary = list(vocabulary)
print ("X :",X)
print ()
print ("Vocabulary :", vocabulary)

X : ['hi', 'hello', 'how are you', 'i am making', 'making', 'working', 'studying', 'see you later', 'bye', 'goodbye']

Vocabulary : ['i', 'how', 'see', 'hi', 'later', 'working', 'studying', 'you', 'are', 'am', 'goodbye', 'making', 'bye', 'hello']


## Create document vectors

In [7]:
# Task 5: Convert text data into array of numbers (bag of words) to be trained by neural network
X_encoded = []

def encode_sentence(sentence):
    sentence = preprocess_data([sentence])[0]
    sentence_encoded = [0] * len(vocabulary)
    for i in range(len(vocabulary)):
        if vocabulary[i] in sentence.split(' '):
            sentence_encoded[i] = 1
    return sentence_encoded

X_encoded = [encode_sentence(sentence) for sentence in X]
print (X_encoded)

[[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], [0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]]


In [8]:
# Task 6: Convert intent text data into array of numbers (bag of words)
classes = list(set(Y))

Y_encoded = []
for data_point in Y:
    data_point_encoded = [0] * len(classes)
    for i in range(len(classes)):
        if classes[i] == data_point:
            data_point_encoded[i] = 1
    Y_encoded.append(data_point_encoded)

print (Y_encoded)

[[0, 0, 1], [0, 0, 1], [0, 0, 1], [1, 0, 0], [1, 0, 0], [1, 0, 0], [1, 0, 0], [0, 1, 0], [0, 1, 0], [0, 1, 0]]


# 4. Create training data and test data

In [9]:
# Task 7 Training & test data
X_train = X_encoded
y_train = Y_encoded
X_test = X_encoded
y_test = Y_encoded

print (y_test)

print(len(X_train))

y_train

[[0, 0, 1], [0, 0, 1], [0, 0, 1], [1, 0, 0], [1, 0, 0], [1, 0, 0], [1, 0, 0], [0, 1, 0], [0, 1, 0], [0, 1, 0]]
10


[[0, 0, 1],
 [0, 0, 1],
 [0, 0, 1],
 [1, 0, 0],
 [1, 0, 0],
 [1, 0, 0],
 [1, 0, 0],
 [0, 1, 0],
 [0, 1, 0],
 [0, 1, 0]]

What does y_train represent? Do you understand the array shown above?

In [10]:
# y_train represents the represent the one-hot encoded intent labels that match up with the data/encoded sentence in x_train.
# yes i understand the array above, it contains a list of lists that the correct one hot encoded labels.

# 5. Model training

Now we will use the training data to train our neural network.

In [11]:
# Task 9: Train ANN
model = Sequential()
model.add(Dense(units=64, activation='sigmoid',
                input_dim=len(X_train[0])))
model.add(Dense(units=len(y_train[0]), activation='softmax'))
model.compile(loss=categorical_crossentropy,
              optimizer=SGD(learning_rate=0.01,
                            momentum=0.9, nesterov=True))
model.fit(np.array(X_train), np.array(y_train), epochs=500, batch_size=16)

Epoch 1/500


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 386ms/step - loss: 1.1475
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 1.1353
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 1.1214
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 1.1083
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 1.0980
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.0910
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.0871
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 1.0857
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 1.0856
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 1.0859
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.0857
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.0848
Epoch 13/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 1.0828
Epoch 14/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.0800
Epoch 15/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 1.0767
Epoch 16/500
1/1 ━━━━━━━━━━━━━

## List down predictions

In [12]:
# Task 10: Calculate Predictions
predictions = [argmax(pred) for pred in model.predict(np.array(X_test))]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


# Model Evaluation

Let's evaluate our model now. We will compare the prediction made by the model and our test data:

In [13]:
# Task 11: Evaluate Model
correct = 0
for i in range(len(predictions)):
    if predictions[i] == argmax(y_test[i]):
        correct += 1

print ("Correct:", correct)
print ("Total:", len(predictions))

Correct: 10
Total: 10


# Testing the chatbot

Let's test the chatbot now! We will input a sentence, and then see what class is predicted by the neural network:

In [14]:
# Task 12: Test
for i in range(3):
    print ("Enter a sentence")
    sentence = input()
    prediction= model.predict(np.array([encode_sentence(sentence)]))
    print (classes[argmax(prediction)])

Enter a sentence
hi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
greeting
Enter a sentence
working
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
busy
Enter a sentence
goodbye
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
bye


# Challenge

We have successfully use neural network to map our input to conversation intent.
Your challenge is to modify the code below so that it loops until the predicted class is 'bye'

In [15]:
# Task 13: Modify the code below so that it loops until the predicted class is 'bye'
while True:
    print ("Enter a sentence")
    sentence = input()
    prediction= model.predict(np.array([encode_sentence(sentence)]))
    #print (classes[argmax(prediction)])
    if (classes[argmax(prediction)] == 'bye'):
        print ('Goodbye! See you next time!')
        break
    elif (classes[argmax(prediction)] == 'busy'):
        print('ah... all the best!')
    elif (classes[argmax(prediction)] == 'greeting'):
        print ('Hi!')

Enter a sentence
hi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Hi!
Enter a sentence
bonjour
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
ah... all the best!
Enter a sentence
working
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
ah... all the best!
Enter a sentence
goodbye
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Goodbye! See you next time!


### Great job! You've successfully created a simple chatbot with neural network! How might you improve the chatbot?
You can improve the chatbot by:
- Adding more training data
- Adding more intent
- Focusing on a particular topic and train the chatbot with many training data in that topic

In-general, I would start by adding more words to the vocab. Then something I might do to help identify sentences better is seperate each word in the sentence and train it based on that and then average them together, with variable weights and balances and that would help improve the accuracy of the model.

### Resource:
https://blog.eduonix.com/internet-of-things/simple-nlp-based-chatbot-python/